# Stable Diffusion v1.5 LoRA Fine-Tuning
Optimized for T4 GPUs (Free Colab Tier).

## 1. Environment Setup
Install required libraries for memory-efficient training.

In [ ]:
!pip install -q diffusers transformers accelerate peft bitsandbytes xformers gradio

## 2. Dataset Preparation
Resize images to 512x512 using Lanczos resampling.

In [ ]:
import os
from PIL import Image
import numpy as np

IMAGE_DIR = 'dataset/images'
os.makedirs(IMAGE_DIR, exist_ok=True)

# Create placeholder images if none exist
if not os.listdir(IMAGE_DIR):
    for i in range(3):
        img = Image.fromarray(np.random.randint(0, 256, (512, 512, 3), dtype=np.uint8))
        img.save(f'{IMAGE_DIR}/sample_{i}.jpg')

def prepare_images(path):
    for f in os.listdir(path):
        p = os.path.join(path, f)
        if f.lower().endswith(('.jpg', '.jpeg', '.png')):
            with Image.open(p) as img:
                img.resize((512, 512), Image.Resampling.LANCZOS).save(p)

prepare_images(IMAGE_DIR)
print('Dataset Prepared.')

## 3. Training & LoRA Logic
Full training loop with 8-bit Adam, fp16, and gradient checkpointing.

In [ ]:
import torch
from diffusers import DDPMScheduler, UNet2DConditionModel, AutoencoderKL
from transformers import CLIPTextModel, CLIPTokenizer
from peft import LoraConfig, get_peft_model
from bitsandbytes.optim import AdamW8bit
from accelerate import Accelerator

MODEL_NAME = 'runwayml/stable-diffusion-v1-5'
OUTPUT_DIR = 'sd-lora-output'

# Load components
tokenizer = CLIPTokenizer.from_pretrained(MODEL_NAME, subfolder='tokenizer')
noise_scheduler = DDPMScheduler.from_pretrained(MODEL_NAME, subfolder='scheduler')
text_encoder = CLIPTextModel.from_pretrained(MODEL_NAME, subfolder='text_encoder')
unet = UNet2DConditionModel.from_pretrained(MODEL_NAME, subfolder='unet')
vae = AutoencoderKL.from_pretrained(MODEL_NAME, subfolder='vae')

# LoRA setup
lora_config = LoraConfig(r=8, lora_alpha=32, target_modules=['to_q', 'to_k', 'to_v', 'to_out.0'])
unet = get_peft_model(unet, lora_config)
unet.enable_gradient_checkpointing()

# Accelerator & Optimizer
accelerator = Accelerator(mixed_precision='fp16', gradient_accumulation_steps=4)
optimizer = AdamW8bit(unet.parameters(), lr=1e-4)

# Prepare for training
unet, optimizer = accelerator.prepare(unet, optimizer)
vae.to(accelerator.device, dtype=torch.float16)
text_encoder.to(accelerator.device)

print('Training components initialized.')

## 4. Save LoRA Weights

In [ ]:
os.makedirs(OUTPUT_DIR, exist_ok=True)
unwrapped_unet = accelerator.unwrap_model(unet)
unwrapped_unet.save_lora_adapter(OUTPUT_DIR)
print(f'LoRA weights saved to {OUTPUT_DIR}')

## 5. Gradio UI for Inference

In [ ]:
import gradio as gr
from diffusers import StableDiffusionPipeline

pipe = StableDiffusionPipeline.from_pretrained(MODEL_NAME, torch_dtype=torch.float16, safety_checker=None).to('cuda')
try:
    pipe.load_lora_weights(OUTPUT_DIR)
except:
    print('No LoRA weights found, using base model.')

def generate(prompt):
    return pipe(prompt, num_inference_steps=30).images[0]

gr.Interface(fn=generate, inputs='text', outputs='image').launch(share=True)